# Parte 1 · Nem todo exemplo é igualmente difícil — gabarito

**Avaliação com habilidade latente · Instituto Kunumi**

Três classificadores com acurácias praticamente iguais. A pergunta é onde cada um erra, e a resposta já tem a forma da matriz de respostas da IRT.

**Execute a célula de preparação primeiro.** Depois siga as tarefas em ordem: complete as lacunas `___` usando as funções de `utils/handson.py` (no Jupyter e no Colab, `help(funcao)` ou Shift + Tab mostram a documentação), execute e responda à pergunta.

In [ ]:
import subprocess
import sys
from pathlib import Path

# No Colab, baixa o repositório do tutorial; localmente, encontra a raiz a partir desta pasta.
if "google.colab" in sys.modules:
    ROOT = Path("/content/latent-ability-ml")
    if not ROOT.exists():
        subprocess.check_call(["git", "clone", "-q", "https://github.com/manuelfjr/latent-ability-ml.git", str(ROOT)])
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils" / "handson.py").exists())

sys.path.insert(0, str(ROOT))
print("Raiz do projeto:", ROOT)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 110,
    "figure.facecolor": "#F0F0F0",
    "axes.facecolor": "#F0F0F0",
    "savefig.facecolor": "#F0F0F0",
    "axes.edgecolor": "#B4ADA4",
    "axes.labelcolor": "#1C2127",
    "axes.titlecolor": "#1C2127",
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#B4ADA4",
    "grid.alpha": 0.35,
    "xtick.color": "#5E5E5E",
    "ytick.color": "#5E5E5E",
    "text.color": "#1C2127",
    "legend.frameon": False,
    "lines.linewidth": 2.6,
    "axes.prop_cycle": mpl.cycler(color=["#F04E44", "#4392D2", "#2FB7CF", "#62CCA3", "#F0D449"]),
})

## Tarefa 1 · Olhe os dados (~2 min)

Gere um problema sintético de classificação binária em duas dimensões com `make_toy_classification_dataset`: 300 pontos, `class_sep=1.4` e `random_state=56`. Complete as lacunas `___` e execute.

**Pergunta.** Antes de treinar qualquer modelo: em que região você espera erros, e por quê?

In [ ]:
from utils.handson import make_toy_classification_dataset, plot_classification_dataset

# 300 pontos, class_sep=1.4, random_state=56
df = make_toy_classification_dataset(n_samples=300, class_sep=1.4, random_state=56)

ax = plot_classification_dataset(df)
ax.set_title("Duas classes")
plt.show()

**Gabarito.** Na faixa em que as duas nuvens se sobrepõem. Ali P(y ∣ x) fica perto de 1/2 e parte do erro é irredutível (erro de Bayes): nenhum classificador acerta esses pontos de forma sistemática.

## Tarefa 2 · Treine três modelos (~3 min)

Treine três classificadores com vieses indutivos diferentes (árvore de decisão, k-vizinhos e regressão logística) com `evaluate_models_on_dataset`, que reserva 30% dos dados (90 exemplos) para teste. Passe o `df` da tarefa anterior e mostre a coluna `"accuracy"` do resumo.

**Pergunta.** A diferença justificaria escolher um dos modelos? O que essa tabela não informa?

In [ ]:
from utils.handson import evaluate_models_on_dataset, summarize_classification_results

# Treina os três classificadores; devolve uma linha por (modelo, exemplo de teste)
resultados = evaluate_models_on_dataset(df, scenario="ita", random_state=56)
print("colunas:", resultados.columns.tolist())

# Uma linha por modelo: mostre só a coluna de acurácia
summarize_classification_results(resultados).set_index("model")[["accuracy"]].round(3)

**Gabarito.** As acurácias ficam entre 0,92 e 0,94. Com 90 exemplos, 2 pontos percentuais são 2 exemplos, dentro do ruído amostral. E um escalar por modelo não diz *quais* exemplos foram errados nem se os modelos erram nos mesmos lugares.

## Tarefa 3 · Quem acertou o quê? (~3 min)

Reorganize o resultado longo como matriz binária com `DataFrame.pivot`: modelos (`model`) nas linhas, exemplos de teste (`example_id`) nas colunas e acerto (`correct`: `1` acerto, `0` erro) nas células. Aparecem as 12 primeiras colunas.

**Pergunta.** Que tipo de coluna ajuda a comparar os modelos, e que tipo não ajuda?

In [ ]:
# Modelos nas linhas, exemplos de teste nas colunas, acerto (1) ou erro (0) nas células
respostas = resultados.pivot(index="model", columns="example_id", values="correct")

print(respostas.shape, "→ (modelos, exemplos de teste)")
respostas.iloc[:, :12]

**Gabarito.** Colunas constantes (todos acertam ou todos erram) não distinguem os modelos; só as colunas mistas carregam informação. Esta matriz respondente × item com respostas dicotômicas é a entrada de um modelo de IRT: os modelos fazem o papel de alunos, e os exemplos, o de questões de uma prova.

## Tarefa 4 · Meça a dificuldade (~3 min)

Uma primeira medida de dificuldade, sem modelo latente: a fração de classificadores que erra cada exemplo (o análogo do índice de dificuldade da teoria clássica dos testes). Escreva-a a partir de `respostas`, em uma linha.

**Pergunta.** Quantos exemplos de fato diferenciam os modelos? O que isso diz sobre o tamanho efetivo do teste?

In [ ]:
dificuldade = 1 - respostas.mean(axis=0)   # fração de modelos que erraram cada exemplo

print("errados por todos:  ", int((dificuldade == 1).sum()))
print("errados por ninguém:", int((dificuldade == 0).sum()))
dificuldade.sort_values(ascending=False).head(8).round(2)

**Gabarito.** 79 dos 90 exemplos são acertados por todos e 2 são errados por todos; sobram 9 exemplos informativos. A comparação entre os modelos é decidida por menos de uma dezena de itens: o tamanho efetivo do teste é muito menor que 90.

## Tarefa 5 · Veja onde eles estão (~3 min)

Destaque, no plano das features, os exemplos de teste que pelo menos um modelo errou. Complete o filtro sobre `dificuldade`; o gráfico já está pronto.

**Pergunta.** A dificuldade parece uma propriedade do exemplo ou ruído? Que limitação tem a medida da tarefa anterior?

In [ ]:
dificeis = dificuldade[dificuldade > 0].index   # exemplos que pelo menos um modelo errou
teste = df[df.example_id.isin(respostas.columns)]
facil = teste[~teste.example_id.isin(dificeis)]
dificil = teste[teste.example_id.isin(dificeis)]

fig, ax = plt.subplots()
for label, cor in [(0, "#B4ADA4"), (1, "#5E5E5E")]:
    grupo = facil[facil.label == label]
    ax.scatter(grupo.feature_1, grupo.feature_2, s=16, color=cor, label=f"classe {label}: todos acertam")
ax.scatter(dificil.feature_1, dificil.feature_2, s=90, color="#F04E44", label="algum modelo erra")
ax.set_title("Onde os modelos erram")
ax.legend()
plt.show()

**Gabarito.** Os exemplos difíceis se concentram na região de sobreposição: dificuldade é uma propriedade do exemplo. A limitação é que a fração de erros trata todos os modelos como equivalentes: errar um exemplo que só o pior modelo erra pesa o mesmo que errar um que derruba o melhor. A IRT resolve isso estimando, ao mesmo tempo, a habilidade de cada modelo e a dificuldade de cada exemplo.

## Desafio · Desafio: deixe o problema mais difícil (~3 min)

Esta célula refaz tudo de uma vez. Mude `class_sep` de `1.4` para `0.6` e execute.

**Pergunta.** O que acontece com o número de exemplos informativos e com a separação entre os modelos?

In [ ]:
class_sep = 0.6   # <- mude aqui

dados = make_toy_classification_dataset(n_samples=300, class_sep=class_sep, random_state=56)
res = evaluate_models_on_dataset(dados, scenario="desafio", random_state=56)
tabela = res.pivot(index="model", columns="example_id", values="correct")

print("exemplos difíceis:", int((tabela.mean(axis=0) < 1).sum()), "de", tabela.shape[1])
tabela.mean(axis=1).round(3).rename("acurácia")

**Gabarito.** Com `class_sep = 0.6`, os exemplos que algum modelo erra passam de 11 para cerca de 36 e as acurácias caem para perto de 0,77. O teste fica mais informativo, mas também mais ruidoso. É nesse regime que um modelo latente compensa: ele separa o que é habilidade do modelo do que é dificuldade do exemplo.